## Stage3 Replay Training
Uses replay CSVs from `stage3_replay_preprocessing.ipynb` to reduce Stage2 forgetting.

In [8]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
import timm

from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, recall_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
UNIFIED_CLASSES = ['MEL','BCC','SCC','AK','NEV','BKL','DF','VASC','SEK','TINEA','PSORIASIS','VITILIGO','MELASMA','FUNGAL','ECZEMA','URTICARIA']
UNIFIED_INDEX = {c: i for i, c in enumerate(UNIFIED_CLASSES)}
PREPROCESSING_CONFIG = {'input_size': 260, 'mean': [0.485,0.456,0.406], 'std': [0.229,0.224,0.225]}

REPLAY_TRAIN = Path(r'stage3_output\stage3_replay_train.csv')
REPLAY_VAL = Path(r'stage3_output\stage3_replay_val.csv')
PAD_MONITOR_VAL = Path(r'stage3_output\pad_monitor_val.csv')

assert REPLAY_TRAIN.exists() and REPLAY_VAL.exists() and PAD_MONITOR_VAL.exists(), 'Run stage3_replay_preprocessing.ipynb first.'
print('Device:', DEVICE)

Device: cuda


In [9]:
train_tf = transforms.Compose([
    transforms.Resize((PREPROCESSING_CONFIG['input_size'], PREPROCESSING_CONFIG['input_size'])),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.25, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(PREPROCESSING_CONFIG['mean'], PREPROCESSING_CONFIG['std']),
    transforms.RandomErasing(p=0.2),
])

eval_tf = transforms.Compose([
    transforms.Resize((PREPROCESSING_CONFIG['input_size'], PREPROCESSING_CONFIG['input_size'])),
    transforms.ToTensor(),
    transforms.Normalize(PREPROCESSING_CONFIG['mean'], PREPROCESSING_CONFIG['std']),
])

class ReplayDataset(Dataset):
    def __init__(self, df: pd.DataFrame, mode='train'):
        self.df = df.reset_index(drop=True).copy()
        self.df['label_idx'] = self.df['unified_label'].map(UNIFIED_INDEX)
        self.df = self.df[self.df['label_idx'].notna()].copy()
        self.df['label_idx'] = self.df['label_idx'].astype(int)
        self.df['fst_group'] = pd.to_numeric(self.df.get('fst_group', 0), errors='coerce').fillna(0).astype(int)
        self.df['dataset_source'] = self.df['dataset_source'].astype(str)
        self.mode = mode
        self.tf = train_tf if mode == 'train' else eval_tf

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        p = Path(r['image_path'])
        try:
            img = Image.open(p).convert('RGB')
        except Exception:
            img = Image.fromarray(np.zeros((260, 260, 3), dtype=np.uint8))
        x = self.tf(img)
        y = int(r['label_idx'])
        fst = int(r['fst_group'])
        src = r['dataset_source']
        return x, y, fst, src

def band_from_fst(v: int):
    if 1 <= v <= 2:
        return 'light'
    if 3 <= v <= 4:
        return 'medium'
    if 5 <= v <= 6:
        return 'dark'
    return 'unknown'

def build_replay_sampler(df: pd.DataFrame):
    class_counts = df['unified_label'].value_counts().to_dict()
    band_counts = df['fst_group'].apply(lambda v: band_from_fst(int(v))).value_counts().to_dict()
    weights = []
    for _, r in df.iterrows():
        c = r['unified_label']
        b = band_from_fst(int(r['fst_group']))
        cw = 1.0 / max(class_counts.get(c, 1), 1)
        bw = 1.0 if b == 'unknown' else (1.0 / max(band_counts.get(b, 1), 1))
        weights.append(cw * bw)
    weights = torch.tensor(weights, dtype=torch.double)
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

In [10]:
train_df = pd.read_csv(REPLAY_TRAIN)
val_df = pd.read_csv(REPLAY_VAL)
pad_val_df = pd.read_csv(PAD_MONITOR_VAL)

train_ds = ReplayDataset(train_df, mode='train')
val_ds = ReplayDataset(val_df, mode='val')
pad_val_ds = ReplayDataset(pad_val_df, mode='val')

sampler = build_replay_sampler(train_ds.df)
train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0)
pad_val_loader = DataLoader(pad_val_ds, batch_size=32, shuffle=False, num_workers=0)

print('train/val/pad_val:', len(train_ds), len(val_ds), len(pad_val_ds))

train/val/pad_val: 26065 4729 324


In [11]:
class EfficientNetB2Unified(nn.Module):
    def __init__(self, num_classes=16):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b2', pretrained=False, num_classes=0, global_pool='')
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.BatchNorm1d(1408),
            nn.Dropout(0.3),
            nn.Linear(1408, num_classes),
        )

    def forward(self, x):
        z = self.backbone.forward_features(x)
        return self.head(z)

model = EfficientNetB2Unified(num_classes=len(UNIFIED_CLASSES)).to(DEVICE)
ckpt2 = torch.load(r'checkpoints\stage2.pth', map_location=DEVICE, weights_only=True)
state = ckpt2['model_state'] if isinstance(ckpt2, dict) and 'model_state' in ckpt2 else ckpt2
missing, unexpected = model.load_state_dict(state, strict=True)
if missing:
    print('[WARN] Missing keys:', missing)
if unexpected:
    print('[WARN] Unexpected keys:', unexpected)

for p in model.parameters():
    p.requires_grad = True

optimizer = Adam([
    {'params': model.backbone.parameters(), 'lr': 5e-6},
    {'params': model.head.parameters(), 'lr': 5e-5},
], weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=12, eta_min=1e-7)

print('Model initialized from checkpoints/stage2.pth')

Model initialized from checkpoints/stage2.pth


In [12]:
def loss_source_aware(logits, targets, sources):
    src = np.array(sources)
    loss_parts = []
    configs = [
        ('dermnet', 0.12),
        ('fitzpatrick17k', 0.04),
        ('isic2019', 0.04),
        ('pad-ufes-20', 0.04),
    ]
    for name, smooth in configs:
        m = torch.from_numpy((np.char.lower(src.astype(str)) == name)).to(logits.device)
        if m.any():
            loss_parts.append(F.cross_entropy(logits[m], targets[m], label_smoothing=smooth))
    if not loss_parts:
        return F.cross_entropy(logits, targets, label_smoothing=0.05)
    return torch.stack(loss_parts).mean()

def run_epoch(loader, train=True):
    model.train(train)
    losses, y_true, y_pred = [], [], []
    for x, y, fst, src in loader:
        x = x.to(DEVICE, dtype=torch.float32)
        y = y.to(DEVICE)

        if train:
            optimizer.zero_grad(set_to_none=True)

        logits = model(x)
        loss = loss_source_aware(logits, y, src)

        if train:
            loss.backward()
            optimizer.step()

        losses.append(float(loss.detach().cpu().item()))
        p = torch.argmax(logits, dim=1)
        y_true.extend(y.detach().cpu().numpy().tolist())
        y_pred.extend(p.detach().cpu().numpy().tolist())

    return {
        'loss': float(np.mean(losses)),
        'acc': float(accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'balanced_acc': float(balanced_accuracy_score(y_true, y_pred)),
    }

def evaluate_pad_monitor(loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y, fst, src in loader:
            x = x.to(DEVICE, dtype=torch.float32)
            y = y.to(DEVICE)
            logits = model(x)
            p = torch.argmax(logits, dim=1)
            y_true.extend(y.detach().cpu().numpy().tolist())
            y_pred.extend(p.detach().cpu().numpy().tolist())
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    idx = {c: i for i, c in enumerate(UNIFIED_CLASSES)}
    return {
        'pad_macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'balanced_acc': float(balanced_accuracy_score(y_true, y_pred)),
        'MEL_recall': float(recall_score(y_true, y_pred, labels=[idx['MEL']], average='macro', zero_division=0)),
        'SCC_recall': float(recall_score(y_true, y_pred, labels=[idx['SCC']], average='macro', zero_division=0)),
        'AK_recall': float(recall_score(y_true, y_pred, labels=[idx['AK']], average='macro', zero_division=0)),
        'fst_dark_f1': 0.0,
    }

def composite(m):
    malignant_avg = (m['MEL_recall'] + m['SCC_recall'] + m['AK_recall']) / 3.0
    return 0.35 * m['pad_macro_f1'] + 0.25 * malignant_avg + 0.20 * m['fst_dark_f1'] + 0.20 * m['balanced_acc']

In [ ]:
best_score = -1.0
EPOCHS = 12

for epoch in range(1, EPOCHS + 1):
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader, train=False)
    pad = evaluate_pad_monitor(pad_val_loader)
    score = composite(pad)

    print(
        f"[Replay Stage3][{epoch:02d}] "
        f"train_f1={tr['macro_f1']:.4f} val_f1={va['macro_f1']:.4f} "
        f"| PAD_f1={pad['pad_macro_f1']:.4f} MEL={pad['MEL_recall']:.4f} SCC={pad['SCC_recall']:.4f} AK={pad['AK_recall']:.4f} "
        f"| score={score:.4f}"
    )

    if score > best_score:
        best_score = score
        torch.save(
            {
                'model_state': model.state_dict(),
                'classes': UNIFIED_CLASSES,
                'stage': '3_replay',
                'best_composite': best_score,
            },
            r'checkpoints\stage3_replay_best.pth'
        )
    scheduler.step()

print('Saved:', r'checkpoints\stage3_replay_best.pth')
print('Best composite:', best_score)

After training, run your locked `evaluate_checkpoint()` on `checkpoints\\stage3_replay_best.pth` and compare with Stage2 on identical loaders.